In [ ]:
import torch

# 再現性を確保するために乱数シードを固定
torch.manual_seed(42)

# -5 から 5 までの範囲を100等分した入力データ x を作成し、(100, 1) の形状に変形
x = torch.linspace(-5, 5, 100).reshape(-1,1)

# 真の関数 y = 3x + 2 に、標準正規分布に従うノイズ（標準偏差0.5）を加算
y = 3*x + 2 + torch.randn_like(x)*0.5

In [ ]:
# 入力データ x の形状を確認（[データ数, 特徴量数] = [100, 1]）
x.shape


torch.Size([100, 1])

In [ ]:
# 目的変数 y の形状を確認（[データ数, 出力数] = [100, 1]）
y.shape

torch.Size([100, 1])

In [ ]:
import torch.nn as nn

# 入力次元1、出力次元1の全結合層（線形結合モデル: y = wx + b）を定義
model = nn.Linear(in_features=1, out_features=1)

# ランダムに初期化された重み (w) とバイアス (b) を表示
print(model.weight)
print(model.bias)

Parameter containing:
tensor([[0.0028]], requires_grad=True)
Parameter containing:
tensor([-0.3721], requires_grad=True)


In [ ]:
# 回帰問題で一般的に用いられる平均二乗誤差（MSE）を損失関数として定義
loss_fn = nn.MSELoss()

In [ ]:
# 確率的勾配降下法（SGD）を用いてパラメータを更新するオプティマイザを定義
# 学習率（learning rate）は 0.01 に設定

optimizer = torch.optim.SGD(
    model.parameters(), 
    lr=0.01
    )

# モデルの学習対象パラメータのジェネレータを確認
model.parameters()

<generator object Module.parameters at 0x11264e420>

In [ ]:
# エポック数（全データを使った学習回数）を設定
epochs = 150

for epoch in range(epochs):

# 1. 順伝播（Forward Pass）: 現在のパラメータで予測値を算出
    pred = model(x)
# 2. 損失（Loss）計算: 予測値と真の値の誤差を評価
    loss = loss_fn(pred, y)
# 3. 勾配の初期化: 前のエポックで計算された勾配をリセット
    optimizer.zero_grad()
# 4. 逆伝播（Backward Pass）: 各パラメータに関する損失の勾配（微分値）を計算
    loss.backward()
# 5. パラメータ更新: 算出された勾配と学習率に基づいて重みとバイアスを更新
    optimizer.step()
# 6. 10エポックごとに損失の値を出力して進捗を確認
    if epoch % 10 == 0:
        print(epoch, loss.item())

0 82.23397827148438
10 5.926093101501465
20 2.8563175201416016
30 1.9585309028625488
40 1.386863350868225
50 1.0058796405792236
60 0.7515482306480408
70 0.5817549228668213
80 0.4683993458747864
90 0.3927225172519684
100 0.3421999216079712
110 0.3084707260131836
120 0.2859528064727783
130 0.2709196209907532
140 0.2608834207057953


In [ ]:
# 学習完了後の重み (w) とバイアス (b) を確認（目標値: w=3, b=2）
print(model.weight)
print(model.bias)

Parameter containing:
tensor([[2.9971]], requires_grad=True)
Parameter containing:
tensor([1.9139], requires_grad=True)


In [ ]:
# パラメータの値が保持されていることを再確認
print(model.weight)
print(model.bias)


Parameter containing:
tensor([[2.9971]], requires_grad=True)
Parameter containing:
tensor([1.9139], requires_grad=True)


In [ ]:
# 重みパラメータ（weight）に残っている直近の勾配（grad）を表示
# 収束していれば 0 に極めて近い値になる

model.weight.grad

tensor([[-8.4462e-06]])

# Day6 Summary

## Learned
- torch.manual_seed() は、PyTorchで乱数（ランダムな数値）を生成する際の「シード値（種）」を指定し、乱数の発生パターンを固定して処理の「再現性」を保つための関数。

なぜ乱数を固定するのか？
コンピュータが生成する乱数は、完全なランダムではなく特定の計算式に基づいて作られる「疑似乱数」。この計算の出発点となる数値を「シード値」と呼び、同じシード値を指定すると毎回必ず全く同じ順序で同じ数字が生成される。

実験結果の再現（再現性の確保）
モデルのパラメータ（重みやバイアス）の初期化、データのシャッフル、ノイズの付加など、学習の過程では至る所で乱数が使われる。シード値を固定しないと実行するたびに初期値が変わり、学習結果（精度や損失の値）がブレてしまう。

正確なデバッグと改善の検証
コードを変更して精度が向上した際、「アルゴリズムの改善による効果」なのか「たまたま良い初期乱数を引いたおかげ」なのかを区別するために、乱数の影響を排除して検証できる。

シード値に渡している 42 という数値自体に特殊な命令的意味はなく、慣習的に広く使われている任意の整数

- torch.linspace は "Linearly Spaced"（線形に均等な間隔） の略で、指定した「開始値」から「終了値」までの間を均等な間隔で指定した個数分だけ分割した数値の列（テンソル）を作る関数
基本的な書き方
torch.linspace(start, end, steps)
start: 開始する数値
end: 終了する数値（※ end の値自体も含まれます）
steps: 生成するデータの総数（要素数）

- torch.randn_like は、指定したテンソルと同じ形状（shape）・データ型・デバイス（CPU/GPU）を持つ、標準正規分布（Standard Normal Distribution: 平均0、標準偏差1）に従う乱数テンソルを作る関数。

randn: 平均 0、標準偏差 1 の正規分布（ガウス分布）からランダムな数値を生成する。

_like: 引数に渡したテンソルと「同じ形状・属性で」作成する。

torch.nn は、PyTorch でニューラルネットワーク（Neural Networks）や機械学習モデルを構築するための基本機能が詰まったモジュール。

nn は Neural Networks の略で、モデルのレイヤー（層）や損失関数など、ディープラーニングに必要な部品が最初から用意されている。

- nn.Module
- nn.Linear
- MSELoss
- SGD
- optimizer.step()
- optimizer.zero_grad()

## Questions

- AdamはSGDと何が違う？
- optimizer.step()は内部で何をしている？